# Import the dependencies 

In [122]:
%pip install rdkit --quiet

Note: you may need to restart the kernel to use updated packages.


In [123]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, Draw, rdMolDescriptors

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import  LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
)

# Loading the data set into a pandas dataframe 

In [124]:
compounds_data = pd.read_csv('./sample_data/tox21.csv')
compounds_data.head()

,NR-AR,NR-AR-LBD,NR-AhR,NR-Aromatase,NR-ER,NR-ER-LBD,NR-PPAR-gamma,SR-ARE,SR-ATAD5,SR-HSE,SR-MMP,SR-p53,mol_id,smiles
0,0.0,0.0,1.0,NaN,NaN,0.0,0.0,1.0,0.0,0.0,0.0,0.0,TOX3021,CCOc1ccc2nc(S(N)(=O)=O)sc2c1
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,TOX3020,CCN1C(=O)NC(c2ccccc2)C1=O
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN,NaN,TOX3024,CC[C@]1(O)CC[C@H]2[C@@H]3CCC4=CCCC[C@@H]4[C@H]...
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,TOX3027,CCCN(CC)C(CC)C(=O)Nc1c(C)cccc1C
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TOX20800,CC(O)(P(=O)(O)O)P(=O)(O)O


In [125]:
# Displaying the last 5 data
compounds_data.tail()

,NR-AR,NR-AR-LBD,NR-AhR,NR-Aromatase,NR-ER,NR-ER-LBD,NR-PPAR-gamma,SR-ARE,SR-ATAD5,SR-HSE,SR-MMP,SR-p53,mol_id,smiles
7826,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN,NaN,TOX2725,CCOc1nc2cccc(C(=O)O)c2n1Cc1ccc(-c2ccccc2-c2nnn...
7827,1.0,1.0,0.0,0.0,1.0,0.0,NaN,NaN,0.0,0.0,NaN,0.0,TOX2370,CC(=O)[C@H]1CC[C@H]2[C@@H]3CCC4=CC(=O)CC[C@]4(...
7828,1.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,TOX2371,C[C@]12CC[C@H]3[C@@H](CCC4=CC(=O)CC[C@@]43C)[C...
7829,1.0,1.0,0.0,NaN,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,TOX2377,C[C@]12CC[C@@H]3c4ccc(O)cc4CC[C@H]3[C@@H]1CC[C...
7830,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,TOX2724,COc1ccc2c(c1OC)CN1CCc3cc4c(cc3C1C2)OCO4


# Exploratory Data Analysis

In [126]:
# Checking the number of rows and columns of the dataset
compounds_data.shape

(7831, 14)

In [127]:
# Getting some info about the dataset
compounds_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7831 entries, 0 to 7830
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   NR-AR          7265 non-null   float64
 1   NR-AR-LBD      6758 non-null   float64
 2   NR-AhR         6549 non-null   float64
 3   NR-Aromatase   5821 non-null   float64
 4   NR-ER          6193 non-null   float64
 5   NR-ER-LBD      6955 non-null   float64
 6   NR-PPAR-gamma  6450 non-null   float64
 7   SR-ARE         5832 non-null   float64
 8   SR-ATAD5       7072 non-null   float64
 9   SR-HSE         6467 non-null   float64
 10  SR-MMP         5810 non-null   float64
 11  SR-p53         6774 non-null   float64
 12  mol_id         7831 non-null   object 
 13  smiles         7831 non-null   object 
dtypes: float64(12), object(2)
memory usage: 856.6+ KB


In [128]:
# Checking for null values in the dataset
compounds_data.isnull().sum()

NR-AR             566
NR-AR-LBD        1073
NR-AhR           1282
NR-Aromatase     2010
NR-ER            1638
NR-ER-LBD         876
NR-PPAR-gamma    1381
SR-ARE           1999
SR-ATAD5          759
SR-HSE           1364
SR-MMP           2021
SR-p53           1057
mol_id              0
smiles              0
dtype: int64

**Note on missingness:** every endpoint column has a different number of NaNs — this is expected in Tox21. A compound simply wasn't tested against every assay. We'll handle this per-endpoint below (mask out NaNs for whichever target we're modeling) rather than dropping rows globally, since dropping globally would throw away perfectly good data for the other 11 endpoints.

# Converting SMILES to Molecular Description

In [129]:
# Creating a function that takes in a SMILE and gives out its molecular properties

def calculate_descriptors(smiles):
    """Convert a SMILES string into a small set of RDKit molecular descriptors.
    Returns None if RDKit fails to parse the molecule (invalid/unsupported SMILES)."""
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    descriptors = {
        "MolWt": Descriptors.ExactMolWt(mol),
        "LogP": Descriptors.MolLogP(mol),
        "TPSA": rdMolDescriptors.CalcTPSA(mol),
        "HBD": rdMolDescriptors.CalcNumHBD(mol),
        "HBA": rdMolDescriptors.CalcNumHBA(mol),
        # A couple of extra descriptors - cheap to add and generally useful
        # for toxicity/bioactivity prediction:
        "RotatableBonds": rdMolDescriptors.CalcNumRotatableBonds(mol),
        "AromaticRings": rdMolDescriptors.CalcNumAromaticRings(mol),
    }

    return descriptors

In [130]:

# Testing the function
calculate_descriptors("CCO")


{'MolWt': 46.041864812,
 'LogP': -0.0014000000000000123,
 'TPSA': 20.23,
 'HBD': 1,
 'HBA': 1,
 'RotatableBonds': 0,
 'AromaticRings': 0}

In [131]:
# Apply to every SMILES in the dataset. RDKit will print warnings for a handful of
# molecules it can't parse (e.g. unusual valence states) - that's expected and those
# rows become None below.
descriptors_data = compounds_data["smiles"].apply(calculate_descriptors)
print(f"Failed to parse: {descriptors_data.isna().sum()} out of {len(descriptors_data)} molecules")

[14:29:29] WARNING: not removing hydrogen atom without neighbors
[14:29:31] Explicit valence for atom # 8 Al, 6, is greater than permitted
[14:29:32] Explicit valence for atom # 3 Al, 6, is greater than permitted
[14:29:32] Explicit valence for atom # 4 Al, 6, is greater than permitted
[14:29:33] Explicit valence for atom # 4 Al, 6, is greater than permitted
[14:29:34] Explicit valence for atom # 9 Al, 6, is greater than permitted
[14:29:34] Explicit valence for atom # 5 Al, 6, is greater than permitted
[14:29:35] Explicit valence for atom # 16 Al, 6, is greater than permitted
[14:29:37] Explicit valence for atom # 20 Al, 6, is greater than permitted


Failed to parse: 8 out of 7831 molecules


In [132]:
# Drop the rows RDKit couldn't parse, and rebuild a clean, aligned dataframe
valid_mask = descriptors_data.notna()
descriptors_df = pd.DataFrame(descriptors_data[valid_mask].tolist(), index=compounds_data.index[valid_mask])

compounds_with_descriptors = pd.concat(
    [compounds_data.loc[valid_mask].reset_index(drop=True),
     descriptors_df.reset_index(drop=True)],
    axis=1
)
print(compounds_with_descriptors.shape)
compounds_with_descriptors.head()

(7823, 21)


,NR-AR,NR-AR-LBD,NR-AhR,NR-Aromatase,NR-ER,NR-ER-LBD,NR-PPAR-gamma,SR-ARE,SR-ATAD5,SR-HSE,...,SR-p53,mol_id,smiles,MolWt,LogP,TPSA,HBD,HBA,RotatableBonds,AromaticRings
0,0.0,0.0,1.0,NaN,NaN,0.0,0.0,1.0,0.0,0.0,...,0.0,TOX3021,CCOc1ccc2nc(S(N)(=O)=O)sc2c1,258.013284,1.34240,82.28,1,5,3,2
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,...,0.0,TOX3020,CCN1C(=O)NC(c2ccccc2)C1=O,204.089878,1.29940,49.41,1,2,2,1
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,...,NaN,TOX3024,CC[C@]1(O)CC[C@H]2[C@@H]3CCC4=CCCC[C@@H]4[C@H]...,288.245316,5.09030,20.23,1,1,1,0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,...,0.0,TOX3027,CCCN(CC)C(CC)C(=O)Nc1c(C)cccc1C,276.220164,3.75244,32.34,1,2,7,1
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,TOX20800,CC(O)(P(=O)(O)O)P(=O)(O)O,205.974526,-0.99220,135.29,5,3,2,0


In [133]:
# Seperating the dataset into features and label
feature_column = ["MolWt", "LogP", "TPSA", "HBD", "HBA", "RotatableBonds", "AromaticRings"]


# 5. Train one model per toxicity endpoint

Instead of repeating the same 6-7 steps manually for each of the 12 endpoints (NR-AR, NR-AhR, SR-p53, ...), we wrap the whole pipeline in a function. This also lets us apply the two important fixes consistently:
- **Scale features** with `StandardScaler`, fit only on the training fold.
- **Balance the classes** with `class_weight='balanced'`, so the model is actually penalized for missing the rare toxic class instead of just learning to predict "non-toxic" every time.


In [134]:
def train_and_evaluate(target, verbose=True):
    """Train a logistic regression toxicity classifier for one Tox21 endpoint.
    Returns a dict of metrics; optionally prints a short report."""

    # Only keep rows where this particular endpoint was actually tested
    mask = compounds_with_descriptors[target].notna()
    X = compounds_with_descriptors.loc[mask, feature_column]
    Y = compounds_with_descriptors.loc[mask, target]

    X_train, X_test, Y_train, Y_test = train_test_split(
        X, Y, test_size=0.2, random_state=24, stratify=Y
    )

    # Fit the scaler on the TRAINING data only, then apply it to both sets.
    # (Fitting on the full X before splitting would leak test-set information.)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = LogisticRegression(max_iter=1000, random_state=2, class_weight="balanced")
    model.fit(X_train_scaled, Y_train)

    predictions = model.predict(X_test_scaled)
    probabilities = model.predict_proba(X_test_scaled)[:, 1]

    test_accuracy = accuracy_score(Y_test, predictions)
    test_auc = roc_auc_score(Y_test, probabilities)

    # 5-fold cross-validated AUC on the training data, so we're not relying on a single split
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=24)
    cv_scores = cross_val_score(
        LogisticRegression(max_iter=1000, random_state=2, class_weight="balanced"),
        scaler.fit_transform(X), Y, cv=cv, scoring="roc_auc"
    )

    if verbose:
        print(f"=== {target} ===")
        print(f"n = {len(Y)}  (positives: {int(Y.sum())}, {Y.mean():.1%})")
        print(f"Test accuracy: {test_accuracy:.3f}")
        print(f"Test ROC-AUC:  {test_auc:.3f}")
        print(f"5-fold CV ROC-AUC: {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}")
        print(classification_report(Y_test, predictions, target_names=["non-toxic", "toxic"]))
        print("Confusion matrix (rows=actual, cols=predicted):")
        print(confusion_matrix(Y_test, predictions))
        print()

    return {
        "target": target,
        "n_samples": len(Y),
        "n_positive": int(Y.sum()),
        "positive_rate": Y.mean(),
        "test_accuracy": test_accuracy,
        "test_auc": test_auc,
        "cv_auc_mean": cv_scores.mean(),
        "cv_auc_std": cv_scores.std(),
        "model": model,
        "scaler": scaler,
    }


In [135]:
# Run it for NR-AR first (your original target) to see the full report
result_nr_ar = train_and_evaluate("NR-AR")


=== NR-AR ===
n = 7258  (positives: 308, 4.2%)
Test accuracy: 0.816
Test ROC-AUC:  0.737
5-fold CV ROC-AUC: 0.754 +/- 0.032
              precision    recall  f1-score   support

   non-toxic       0.98      0.83      0.90      1390
       toxic       0.12      0.53      0.20        62

    accuracy                           0.82      1452
   macro avg       0.55      0.68      0.55      1452
weighted avg       0.94      0.82      0.87      1452

Confusion matrix (rows=actual, cols=predicted):
[[1152  238]
 [  29   33]]



# 6. Run every endpoint and compare

Now that the pipeline is a function, we can loop over all 12 endpoints instead of copy-pasting cells.

In [136]:
toxicity_targets = [
    "NR-AR", "NR-AR-LBD", "NR-AhR", "NR-Aromatase", "NR-ER", "NR-ER-LBD",
    "NR-PPAR-gamma", "SR-ARE", "SR-ATAD5", "SR-HSE", "SR-MMP", "SR-p53",
]

all_results = [train_and_evaluate(t, verbose=False) for t in toxicity_targets]

summary = pd.DataFrame(all_results).drop(columns=["model", "scaler"])
summary = summary.sort_values("test_auc", ascending=False).reset_index(drop=True)
summary


,target,n_samples,n_positive,positive_rate,test_accuracy,test_auc,cv_auc_mean,cv_auc_std
0,SR-MMP,5804,918,0.158167,0.760551,0.827391,0.839574,0.013175
1,NR-AhR,6542,768,0.117395,0.737204,0.798797,0.819654,0.019016
2,NR-Aromatase,5815,300,0.051591,0.737747,0.785978,0.798465,0.021237
3,SR-p53,6767,423,0.062509,0.680207,0.774046,0.752995,0.021139
4,NR-ER-LBD,6948,349,0.050230,0.700000,0.758712,0.727476,0.029743
5,NR-AR-LBD,6751,237,0.035106,0.793486,0.756543,0.768502,0.035682
6,NR-AR,7258,308,0.042436,0.816116,0.737398,0.753519,0.031585
7,SR-ARE,5825,942,0.161717,0.669528,0.722941,0.701315,0.017831
8,SR-ATAD5,7065,264,0.037367,0.687190,0.706638,0.698602,0.026071
9,SR-HSE,6460,372,0.057585,0.659443,0.686992,0.700445,0.012525
